# 🧩 Mini-Lab: Async LLM Calls

**Module 10: Production Deployment** | **Type: Mini-Lab (Brick)**

---

## Learning Objectives

By the end of this mini-lab, you will be able to:

1. **Understand** why async matters for LLM applications that make I/O-bound API calls
2. **Implement** async LLM calls using the OpenAI async client
3. **Compare** sequential vs. concurrent execution times for multiple LLM requests
4. **Apply** proper error handling in async contexts

## Target Concepts

| Concept | Description |
|---------|-------------|
| Async Patterns | Using `async`/`await` and `asyncio.gather` to make concurrent LLM calls, maximizing throughput for I/O-bound operations |
| Error Handling | Gracefully catching and handling errors in async code so one failed call doesn't crash the entire batch |

## Setup

In [2]:
import asyncio
import time
from openai import OpenAI, AsyncOpenAI
from dotenv import load_dotenv
from IPython.display import display, Markdown

load_dotenv()

def md(text):
    """Display text as rendered markdown."""
    display(Markdown(text))

MODEL = "gpt-4o-mini"
print("✅ Ready")

✅ Ready


## Why Async Matters for LLM Apps

LLM API calls are **I/O-bound** — your code spends most of its time *waiting* for the remote server to respond. During that wait, a synchronous program sits idle.

With `async`, Python can **start multiple requests and wait for them all at once**, rather than one-by-one:

```
Synchronous (sequential):
  Request 1 ████████░░░░░░░░  (2s)
  Request 2 ░░░░░░░░████████  (2s)
  Total: ~4s

Async (concurrent):
  Request 1 ████████          (2s)
  Request 2 ████████          (2s)  ← runs at the same time!
  Total: ~2s
```

This is especially valuable when your API server handles many users or when you need to make several LLM calls for a single request (e.g., parallel evaluations, fan-out queries).

## Step 1 — Sequential (Sync) Calls

Let's establish a baseline. We'll ask the LLM three different questions **one at a time** using the standard synchronous client.

In [3]:
sync_client = OpenAI()

questions = [
    "What is an API gateway? Reply in one sentence.",
    "What is rate limiting? Reply in one sentence.",
    "What is load balancing? Reply in one sentence.",
]

start = time.perf_counter()

sync_results = []
for q in questions:
    resp = sync_client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": q}],
        temperature=0,
    )
    sync_results.append(resp.choices[0].message.content)

sync_time = time.perf_counter() - start

for q, a in zip(questions, sync_results):
    print(f"Q: {q}")
    print(f"A: {a}\n")

print(f"⏱️  Sequential time: {sync_time:.2f}s")

Q: What is an API gateway? Reply in one sentence.
A: An API gateway is a server that acts as an intermediary between clients and backend services, managing requests, routing, and providing features like authentication, rate limiting, and logging.

Q: What is rate limiting? Reply in one sentence.
A: Rate limiting is a technique used to control the amount of incoming or outgoing traffic to or from a network, API, or service by restricting the number of requests a user can make in a given time period.

Q: What is load balancing? Reply in one sentence.
A: Load balancing is the process of distributing network or application traffic across multiple servers to ensure optimal resource use, minimize response time, and prevent overload on any single server.

⏱️  Sequential time: 4.41s


## Step 2 — Concurrent (Async) Calls

Now let's make the **same three calls concurrently** using the `AsyncOpenAI` client and `asyncio.gather`.

Key pieces:
- **`AsyncOpenAI()`** — an async version of the OpenAI client
- **`async def`** — defines a coroutine (a function that can be paused/resumed)
- **`await`** — pauses the coroutine until the result is ready, letting other coroutines run
- **`asyncio.gather(*tasks)`** — runs multiple coroutines concurrently and collects their results

In [4]:
async_client = AsyncOpenAI()


async def ask(question: str) -> str:
    """Send a single question to the LLM asynchronously."""
    resp = await async_client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": question}],
        temperature=0,
    )
    return resp.choices[0].message.content


async def ask_all(questions: list[str]) -> list[str]:
    """Send all questions concurrently."""
    tasks = [ask(q) for q in questions]
    return await asyncio.gather(*tasks)


start = time.perf_counter()
async_results = await ask_all(questions)
async_time = time.perf_counter() - start

for q, a in zip(questions, async_results):
    print(f"Q: {q}")
    print(f"A: {a}\n")

print(f"⏱️  Concurrent time: {async_time:.2f}s")

Q: What is an API gateway? Reply in one sentence.
A: An API gateway is a server that acts as an intermediary between clients and backend services, managing requests, routing, and providing features like authentication, rate limiting, and logging.

Q: What is rate limiting? Reply in one sentence.
A: Rate limiting is a technique used to control the amount of incoming or outgoing traffic to or from a network, API, or service by restricting the number of requests a user can make in a given time period.

Q: What is load balancing? Reply in one sentence.
A: Load balancing is the process of distributing network or application traffic across multiple servers to ensure optimal resource use, minimize response time, and prevent overload on any single server.

⏱️  Concurrent time: 1.29s


## Step 3 — Compare the Times

In [5]:
speedup = sync_time / async_time if async_time > 0 else float("inf")

md(
    f"| Approach | Time | Speedup |\n"
    f"|----------|------|---------|\n"
    f"| Sequential (sync) | {sync_time:.2f}s | 1.0x |\n"
    f"| Concurrent (async) | {async_time:.2f}s | **{speedup:.1f}x** |"
)

| Approach | Time | Speedup |
|----------|------|---------|
| Sequential (sync) | 4.41s | 1.0x |
| Concurrent (async) | 1.29s | **3.4x** |

With 3 concurrent calls, you should see roughly a **2–3x speedup**. The more independent calls you need to make, the bigger the benefit.

> **Important:** Async doesn't make individual calls faster — it makes *waiting for multiple calls* faster by overlapping them.

## Step 4 — Error Handling in Async Code

When running calls concurrently, **one failure shouldn't crash everything**. By default, `asyncio.gather` propagates the first exception and cancels remaining tasks. We need a strategy to handle errors gracefully.

Two approaches:

1. **Wrap each coroutine in try/except** — handle errors per-task
2. **Use `return_exceptions=True`** — gather returns exceptions as values instead of raising them

Let's see both.

### Approach 1 — Try/Except Per Task

This gives you full control over what happens when a single call fails.

In [6]:
async def safe_ask(question: str) -> dict:
    """Ask with per-task error handling. Returns a dict with status."""
    try:
        resp = await async_client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "user", "content": question}],
            temperature=0,
        )
        return {"status": "success", "question": question, "answer": resp.choices[0].message.content}
    except Exception as e:
        return {"status": "error", "question": question, "error": str(e)}


# Mix in a call that will fail (bad model name)
async def bad_ask() -> dict:
    """This call uses a non-existent model to simulate a failure."""
    try:
        resp = await async_client.chat.completions.create(
            model="non-existent-model-xyz",
            messages=[{"role": "user", "content": "Hello"}],
        )
        return {"status": "success", "answer": resp.choices[0].message.content}
    except Exception as e:
        return {"status": "error", "error": str(e)}


results = await asyncio.gather(
    safe_ask("What is caching? Reply in one sentence."),
    bad_ask(),  # This one will fail
    safe_ask("What is a webhook? Reply in one sentence."),
)

for i, r in enumerate(results, 1):
    if r["status"] == "success":
        print(f"Task {i} ✅: {r['answer']}")
    else:
        print(f"Task {i} ❌: {r['error'][:80]}...")

Task 1 ✅: Caching is the process of storing copies of frequently accessed data in a temporary storage location to improve retrieval speed and reduce latency.
Task 2 ❌: Error code: 404 - {'error': {'message': 'The model `non-existent-model-xyz` does...
Task 3 ✅: A webhook is a user-defined HTTP callback that allows one application to send real-time data to another application when a specific event occurs.


Notice: tasks 1 and 3 **succeeded** even though task 2 failed. Each task handled its own error independently.

### Approach 2 — `return_exceptions=True`

A quicker approach — `asyncio.gather` returns exception objects in the results list instead of raising them.

In [7]:
async def ask_raw(question: str, model: str = MODEL) -> str:
    """Plain async call — no internal error handling."""
    resp = await async_client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": question}],
        temperature=0,
    )
    return resp.choices[0].message.content


results = await asyncio.gather(
    ask_raw("What is DNS? Reply in one sentence."),
    ask_raw("Hello", model="non-existent-model-xyz"),  # Will fail
    ask_raw("What is TLS? Reply in one sentence."),
    return_exceptions=True,  # ← Key: exceptions become return values
)

for i, r in enumerate(results, 1):
    if isinstance(r, Exception):
        print(f"Task {i} ❌: {type(r).__name__}: {str(r)[:80]}...")
    else:
        print(f"Task {i} ✅: {r}")

Task 1 ✅: DNS, or Domain Name System, is a hierarchical system that translates human-readable domain names into IP addresses, allowing browsers to locate and access websites on the internet.
Task 2 ❌: NotFoundError: Error code: 404 - {'error': {'message': 'The model `non-existent-model-xyz` does...
Task 3 ✅: TLS, or Transport Layer Security, is a cryptographic protocol designed to provide secure communication over a computer network by encrypting data and ensuring privacy and integrity between client and server.


### Which Approach to Use?

| Approach | Best For |
|----------|----------|
| **Try/except per task** | When you need custom fallback logic per call (e.g., return a default answer) |
| **`return_exceptions=True`** | When you want to collect all results/errors and process them after |

## 🎯 Summary

### Key Takeaways

1. **Async concurrency** — `AsyncOpenAI` + `asyncio.gather` lets you run multiple LLM calls at the same time, dramatically reducing total wait time for I/O-bound operations
2. **`asyncio.gather`** — collects multiple coroutines and runs them concurrently; results come back in the same order as the input tasks
3. **Per-task error handling** — wrapping each coroutine in `try/except` ensures one failure doesn't take down the entire batch
4. **`return_exceptions=True`** — a convenient alternative that returns exceptions as values in the results list instead of raising them